# 🇧🇩 Bengali → English Translator
Uses **Google Translate** (via `deep-translator`) — **no API key needed**.

Reads a Bengali `.txt` from Google Drive, translates it to English **preserving all spacing and formatting**, and saves the result as a new `.txt` file.

**Fixes applied:**
- Char limit lowered to 2,000 (Google Translate's real-world safe limit)
- Retry with exponential backoff on connection errors
- Longer delay between calls to avoid rate-limiting

In [ ]:
# ─── Step 1: Install deep-translator ─────────────────────────────────────────
!pip install deep-translator -q
print('✅ deep-translator installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.6 MB/s eta 0:00:00
✅ deep-translator installed


In [ ]:
# ─── Step 2: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted at /content/drive')

Mounted at /content/drive
✅ Drive mounted at /content/drive


In [ ]:
# ─── Step 3: Configuration — EDIT THESE VALUES ───────────────────────────────

# Full path to your Bengali .txt file inside Google Drive
INPUT_FILE  = "/content/drive/MyDrive/budha.txt"          # ← change to your file path

# Where to save the English translation
OUTPUT_FILE = "/content/drive/MyDrive/budha_english.txt"  # ← change if you like

print(f"Input  : {INPUT_FILE}")
print(f"Output : {OUTPUT_FILE}")

Input  : /content/drive/MyDrive/budha.txt
Output : /content/drive/MyDrive/budha_english.txt


In [ ]:
# ─── Step 4: Translation ─────────────────────────────────────────────────────
import time
from deep_translator import GoogleTranslator

translator = GoogleTranslator(source='bn', target='en')

# FIX 1: 2,000 chars is the real-world safe limit for Google Translate.
# The advertised 5,000 limit causes RequestError on long Bengali lines.
MAX_CHARS  = 2000
MAX_RETRIES = 4        # FIX 2: retry failed requests
BASE_DELAY  = 1.0      # seconds between every call (avoids rate-limiting)

# ── Read the source file ─────────────────────────────────────────────────────
for enc in ('utf-8', 'utf-8-sig', 'utf-16', 'cp1252'):
    try:
        with open(INPUT_FILE, encoding=enc) as f:
            raw_text = f.read()
        print(f'✅ File read with encoding: {enc}')
        break
    except (UnicodeDecodeError, FileNotFoundError) as e:
        last_error = e
else:
    raise RuntimeError(f'Could not read file: {last_error}')

lines = raw_text.split('\n')
print(f'Total lines (including blanks): {len(lines)}')

# ── Split a long string into parts that each fit within MAX_CHARS ────────────
def split_into_parts(text: str) -> list:
    """Split on Bengali full-stop (।) or '. ' so each part <= MAX_CHARS."""
    if len(text) <= MAX_CHARS:
        return [text]
    parts, current = [], ''
    delimited = text.replace('।', '।<S>').replace('. ', '. <S>')
    for segment in delimited.split('<S>'):
        if len(current) + len(segment) > MAX_CHARS:
            if current:
                parts.append(current.strip())
            current = segment
        else:
            current += segment
    if current:
        parts.append(current.strip())
    return [p for p in parts if p]

# ── Translate one string with retry + exponential backoff ────────────────────
def translate_with_retry(text: str) -> str:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return translator.translate(text)
        except Exception as e:
            if attempt == MAX_RETRIES:
                raise
            wait = BASE_DELAY * (2 ** attempt)  # 2s, 4s, 8s …
            print(f'      ↻ Retry {attempt}/{MAX_RETRIES - 1} after {wait:.0f}s ({e})')
            time.sleep(wait)

# ── Translate a full line (splitting if needed) ──────────────────────────────
def translate_line(text: str) -> str:
    parts = split_into_parts(text)
    translated_parts = []
    for part in parts:
        translated_parts.append(translate_with_retry(part))
        if len(parts) > 1:
            time.sleep(BASE_DELAY)  # extra delay between sub-parts
    return ' '.join(translated_parts)

# ── Main loop ────────────────────────────────────────────────────────────────
translated_lines = []
errors = []

for i, line in enumerate(lines):
    stripped = line.strip()

    if not stripped:
        translated_lines.append('')   # preserve blank lines
        continue

    try:
        translated = translate_line(stripped)
        translated_lines.append(translated)
        if (i + 1) % 10 == 0:
            print(f'  Line {i+1}/{len(lines)} ✓')
    except Exception as e:
        print(f'  ⚠️  Line {i+1} failed after {MAX_RETRIES} retries: {e} — keeping original')
        translated_lines.append(line)
        errors.append(i + 1)

    time.sleep(BASE_DELAY)  # polite delay after every line

print(f'\n✅ Translation done  ({len(lines)} lines processed)')
if errors:
    print(f'⚠️  Lines kept as original: {errors}')
else:
    print('🎉 No errors — all lines translated!')

# ── Write output ─────────────────────────────────────────────────────────────
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    f.write('\n'.join(translated_lines))

print(f'\n📄 Saved to: {OUTPUT_FILE}')

✅ File read with encoding: utf-8
Total lines (including blanks): 131
  Line 30/131 ✓
  Line 40/131 ✓
  Line 50/131 ✓
  Line 60/131 ✓
  Line 70/131 ✓
  Line 120/131 ✓

✅ Translation done  (131 lines processed)
🎉 No errors — all lines translated!

📄 Saved to: /content/drive/MyDrive/budha_english.txt


In [ ]:
# ─── Step 5 (Optional): Preview the first 30 lines of the output ─────────────
with open(OUTPUT_FILE, encoding='utf-8') as f:
    preview = f.readlines()[:30]

print('=' * 60)
print('PREVIEW — first 30 lines of translated file')
print('=' * 60)
for i, line in enumerate(preview, 1):
    print(f'{i:3}│ {line}', end='')
print('\n' + '=' * 60)

PREVIEW — first 30 lines of translated file
  1│ Novel : Introduction a. Concept and Definition of Novel A novel is a story written in prose. People love to tell stories. It has been going on since ancient times. People used to tell stories by word of mouth There was no way to write it down. Later, the novel appeared in its continuation. But the novel is not just a story, it is a kind of creative work. Human life is the source of novels. Writers write novels by mixing their thoughts and imagination with the events that happen in this life. In the novel, we get the expression of the novelist's feelings. This is how the novel has become a kind of creative work. When people then learn to write, the novel appears. Storytellers became extinct after the advent of the printing press in the fifteenth century. The era of modern novel begins. First this modern novel started in Europe, then it spread all over the world. The writing of modern novels began.
  2│ 
  3│ As I said at the beginning, th